# Demand Uncertainty Analysis
This notebook tests how sensitive each surplus pathway is to demand level uncertainty.

**Design:**
- 4 scenarios: baseline, battery, hydrogen, big user (2 nodes)
- 3 demand levels: low (×0.8), mid (×1.0), high (×1.2)
- 12 model runs total
- Output: heatmap of curtailment rate + cost table

## 1. Setup

In [11]:
import calliope
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RESULTS_DIR = Path('results/demand_uncertainty')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEMAND_CSV  = Path('model/input_data_tables/demand.csv')
DEMAND_LOW  = Path('model/input_data_tables/demand_low.csv')
DEMAND_HIGH = Path('model/input_data_tables/demand_high.csv')

MULTIPLIERS = {'low': 0.8, 'mid': 1.0, 'high': 1.2}

SCENARIOS = {
    'baseline':                      'Baseline',
    'surplus_to_battery':            'Battery',
    'surplus_to_hydrogen':           'Hydrogen',
    'surplus_to_big_user_two_nodes': 'Industrial Partner',
}

print('Setup complete.')

Setup complete.


## 2. Generate scaled demand CSV files

In [2]:
# Read original demand (skip header row with 'techs' label)
df_demand = pd.read_csv(DEMAND_CSV, index_col=0)

# Row 0 is the 'techs' header row — keep it, scale only numeric rows
header_row = df_demand.iloc[[0]]           # 'demand_power' label row
data_rows  = df_demand.iloc[1:].astype(float)

# Low demand (x0.8)
df_low = pd.concat([header_row, data_rows * 0.8])
df_low.to_csv(DEMAND_LOW)

# High demand (x1.2)
df_high = pd.concat([header_row, data_rows * 1.2])
df_high.to_csv(DEMAND_HIGH)

print(f'Original mean demand : {data_rows["NLD"].mean():,.0f} MW')
print(f'Low demand mean      : {(data_rows["NLD"] * 0.8).mean():,.0f} MW  (x0.8)')
print(f'High demand mean     : {(data_rows["NLD"] * 1.2).mean():,.0f} MW  (x1.2)')
print('Saved demand_low.csv and demand_high.csv')

Original mean demand : 36,192 MW
Low demand mean      : 28,954 MW  (x0.8)
High demand mean     : 43,431 MW  (x1.2)
Saved demand_low.csv and demand_high.csv


## 3. Add demand overrides to uncertainties.yaml

In [3]:
UNCERTAINTIES_PATH = Path('model/uncertainties.yaml')

new_overrides = """
  demand_low:
    data_tables:
      demand:
        data: input_data_tables/demand_low.csv

  demand_high:
    data_tables:
      demand:
        data: input_data_tables/demand_high.csv
"""

existing = UNCERTAINTIES_PATH.read_text(encoding='utf-8')

if 'demand_low' not in existing:
    with open(UNCERTAINTIES_PATH, 'a', encoding='utf-8') as f:
        f.write(new_overrides)
    print('Added demand_low and demand_high overrides to uncertainties.yaml')
else:
    print('Overrides already present in uncertainties.yaml — skipping.')

Added demand_low and demand_high overrides to uncertainties.yaml


## 4. Run all 12 model solves
**This cell takes ~10–20 minutes. Results are cached as .nc files so you only need to run it once.**

In [12]:
# demand_level -> override name (mid = no override, use default demand.csv)
DEMAND_OVERRIDES = {
    'low':  'demand_low',
    'mid':  None,
    'high': 'demand_high',
}

for sc_key in SCENARIOS:
    for demand_level, demand_override in DEMAND_OVERRIDES.items():
        out_nc = RESULTS_DIR / f'{sc_key}_{demand_level}.nc'

        if out_nc.exists():
            print(f'  SKIP (exists): {out_nc.name}')
            continue

        # Build scenario string for calliope
        if demand_override:
            # Calliope accepts comma-separated override names as scenario
            # We pass the base scenario overrides + demand override
            # We load with scenario= base scenario, then apply demand override manually
            m = calliope.read_yaml(
                'model/model.yaml',
                scenario=sc_key,
                override_dict={
                    'data_tables': {
                        'demand': {
                            'data': f'input_data_tables/demand_{demand_level}.csv'
                        }
                    }
                }
            )
        else:
            m = calliope.read_yaml('model/model.yaml', scenario=sc_key)

        print(f'  Solving: {sc_key} | demand={demand_level} ...', end=' ', flush=True)
        m.build()
        m.solve()
        status = m.results.attrs.get('termination_condition', 'unknown')
        print(status)
        m.to_netcdf(str(out_nc))

print('\nAll runs complete.')

  SKIP (exists): baseline_low.nc
  SKIP (exists): baseline_mid.nc
  SKIP (exists): baseline_high.nc
  SKIP (exists): surplus_to_battery_low.nc
  SKIP (exists): surplus_to_battery_mid.nc
  SKIP (exists): surplus_to_battery_high.nc
  SKIP (exists): surplus_to_hydrogen_low.nc
  SKIP (exists): surplus_to_hydrogen_mid.nc
  SKIP (exists): surplus_to_hydrogen_high.nc
  SKIP (exists): surplus_to_big_user_two_nodes_low.nc
  SKIP (exists): surplus_to_big_user_two_nodes_mid.nc
  SKIP (exists): surplus_to_big_user_two_nodes_high.nc

All runs complete.


## 5. Extract KPIs from all runs

In [5]:
def _wind_avail(m):
    p = m.inputs.source_use_equals if 'source_use_equals' in m.inputs.data_vars \
        else m.inputs.source_use_max
    cap = m.inputs.flow_cap.sel(techs='wind_offshore', nodes='NLD')
    return float((p.sel(techs='wind_offshore', nodes='NLD') * cap).sum().item())

def _sum(m, var, tech, carrier):
    if var not in m.results.data_vars: return 0.0
    da = m.results[var]
    if 'techs' not in da.dims or tech not in da.coords['techs'].values.tolist(): return 0.0
    da = da.sel(techs=tech)
    if 'carriers' in da.dims:
        if carrier not in da.coords['carriers'].values.tolist(): return 0.0
        da = da.sel(carriers=carrier)
    return float(da.fillna(0).sum().item())

def _total_cost(m):
    if 'cost' not in m.results.data_vars: return np.nan
    return float(m.results.cost.sel(costs='monetary').fillna(0).sum().item())

records = []
for sc_key, sc_label in SCENARIOS.items():
    for demand_level in ['low', 'mid', 'high']:
        nc = RESULTS_DIR / f'{sc_key}_{demand_level}.nc'
        if not nc.exists():
            print(f'  Missing: {nc.name}'); continue

        m = calliope.read_netcdf(str(nc))
        wind_avail = _wind_avail(m)
        curtailed  = _sum(m, 'flow_in',  'curtailment',   'power')
        curt_rate  = curtailed / wind_avail * 100 if wind_avail > 0 else 0.0
        total_cost = _total_cost(m)
        ccgt_gen   = _sum(m, 'flow_out', 'ccgt',          'power')
        load_shed  = _sum(m, 'flow_out', 'load_shedding', 'power')
        sp = m.results.shadow_price_system_balance.sel(nodes='NLD', carriers='power').to_series().dropna()
        neg_price_hours = int((sp < 0).sum())

        records.append({
            'scenario_key':    sc_key,
            'scenario':        sc_label,
            'demand_level':    demand_level,
            'curtailment_TWh': round(curtailed / 1e6, 2),
            'curt_rate_pct':   round(curt_rate, 1),
            'total_cost_MEUR': round(total_cost / 1e3, 1),
            'ccgt_TWh':        round(ccgt_gen / 1e6, 2),
            'load_shed_MWh':   round(load_shed, 0),
            'neg_price_hours': neg_price_hours,
        })

df_all = pd.DataFrame(records)
print(df_all.to_string(index=False))

                 scenario_key           scenario demand_level  curtailment_TWh  curt_rate_pct  total_cost_MEUR  ccgt_TWh  load_shed_MWh  neg_price_hours
                     baseline           Baseline          low           166.72           42.8           2981.1     31.27            0.0             6625
                     baseline           Baseline          mid           123.26           31.7           4732.9     51.22            0.0             5865
                     baseline           Baseline         high            87.10           22.4           7150.3     78.44        15442.0             4875
           surplus_to_battery            Battery          low           136.87           35.2           2078.2     21.57            0.0             6781
           surplus_to_battery            Battery          mid            93.99           24.2           3659.1     39.61            0.0             5966
           surplus_to_battery            Battery         high            57.51    

## 6. Heatmap — Curtailment Rate (%)

In [13]:
DEMAND_ORDER   = ['low', 'mid', 'high']
DEMAND_LABELS  = {'low': 'Low demand\n(×0.8)', 'mid': 'Mid demand\n(×1.0)', 'high': 'High demand\n(×1.2)'}
SCENARIO_ORDER = list(SCENARIOS.keys())
SCENARIO_LABELS= list(SCENARIOS.values())

# Build z matrix: rows=scenarios, cols=demand levels
z_curt = []
text_curt = []
for sc_key in SCENARIO_ORDER:
    row_z, row_t = [], []
    for dl in DEMAND_ORDER:
        match = df_all[(df_all.scenario_key == sc_key) & (df_all.demand_level == dl)]
        val = match['curt_rate_pct'].values[0] if len(match) else np.nan
        row_z.append(val)
        row_t.append(f'{val:.1f}%' if not np.isnan(val) else '—')
    z_curt.append(row_z)
    text_curt.append(row_t)

fig_curt = go.Figure(data=go.Heatmap(
    z=z_curt,
    x=[DEMAND_LABELS[d] for d in DEMAND_ORDER],
    y=SCENARIO_LABELS,
    text=text_curt,
    texttemplate='%{text}',
    textfont={'size': 14},
    colorscale='RdYlGn_r',   # red=high curtailment, green=low
    colorbar=dict(title='Curtailment<br>rate (%)', ticksuffix='%'),
    hoverongaps=False,
))

fig_curt.update_layout(
    title=dict(text='Curtailment Rate (%) by Scenario and Demand Level', font=dict(size=16)),
    xaxis=dict(title='Demand level', tickfont=dict(size=13)),
    yaxis=dict(title='Scenario', tickfont=dict(size=13)),
    width=700, height=420,
    margin=dict(l=160, r=60, t=70, b=60),
)

fig_curt.show()
fig_curt.write_html(str(RESULTS_DIR / 'heatmap_curtailment_rate.html'))
print('Saved: results/demand_uncertainty/heatmap_curtailment_rate.html')

Saved: results/demand_uncertainty/heatmap_curtailment_rate.html


## 7. Heatmap — Total System Cost (M EUR)

In [14]:
z_cost = []
text_cost = []
for sc_key in SCENARIO_ORDER:
    row_z, row_t = [], []
    for dl in DEMAND_ORDER:
        match = df_all[(df_all.scenario_key == sc_key) & (df_all.demand_level == dl)]
        val = match['total_cost_MEUR'].values[0] if len(match) else np.nan
        row_z.append(val)
        row_t.append(f'{val:,.0f}' if not np.isnan(val) else '—')
    z_cost.append(row_z)
    text_cost.append(row_t)

fig_cost = go.Figure(data=go.Heatmap(
    z=z_cost,
    x=[DEMAND_LABELS[d] for d in DEMAND_ORDER],
    y=SCENARIO_LABELS,
    text=text_cost,
    texttemplate='%{text} M€',
    textfont={'size': 13},
    colorscale='RdYlGn_r',   # red=expensive, green=cheap
    colorbar=dict(title='Total cost<br>(M EUR)'),
    hoverongaps=False,
))

fig_cost.update_layout(
    title=dict(text='Total System Cost (M EUR) by Scenario and Demand Level', font=dict(size=16)),
    xaxis=dict(title='Demand level', tickfont=dict(size=13)),
    yaxis=dict(title='Scenario', tickfont=dict(size=13)),
    width=700, height=420,
    margin=dict(l=160, r=60, t=70, b=60),
)

fig_cost.show()
fig_cost.write_html(str(RESULTS_DIR / 'heatmap_cost.html'))
print('Saved: results/demand_uncertainty/heatmap_cost.html')

Saved: results/demand_uncertainty/heatmap_cost.html


## 8. Summary table

In [8]:
pivot = df_all.pivot_table(
    index='scenario',
    columns='demand_level',
    values=['curt_rate_pct', 'total_cost_MEUR', 'neg_price_hours', 'ccgt_TWh'],
    aggfunc='first'
)[['curt_rate_pct', 'total_cost_MEUR', 'neg_price_hours', 'ccgt_TWh']]

# Reorder demand level columns
pivot = pivot.reindex(columns=pd.MultiIndex.from_product(
    [['curt_rate_pct', 'total_cost_MEUR', 'neg_price_hours', 'ccgt_TWh'],
     ['low', 'mid', 'high']]
))

pivot.columns = [
    f'{metric} | {dl}'
    for metric, dl in pivot.columns
]

pivot.index.name = 'Scenario'
display(pivot)

df_all.to_csv(RESULTS_DIR / 'demand_uncertainty_results.csv', index=False)
print('Saved: results/demand_uncertainty/demand_uncertainty_results.csv')

,curt_rate_pct | low,curt_rate_pct | mid,curt_rate_pct | high,total_cost_MEUR | low,total_cost_MEUR | mid,total_cost_MEUR | high,neg_price_hours | low,neg_price_hours | mid,neg_price_hours | high,ccgt_TWh | low,ccgt_TWh | mid,ccgt_TWh | high
Scenario,,,,,,,,,,,,
Baseline,42.8,31.7,22.4,2981.1,4732.9,7150.3,6625,5865,4875,31.27,51.22,78.44
Battery,35.2,24.2,14.8,2078.2,3659.1,5793.9,6781,5966,5109,21.57,39.61,63.73
Big User (2 nodes),36.1,25.8,17.6,1668.5,3587.2,6211.5,6226,5353,4309,31.27,51.22,78.44
Hydrogen,16.7,10.1,5.8,574.2,2745.0,5621.6,4531,3335,2239,31.27,51.22,78.44


Saved: results/demand_uncertainty/demand_uncertainty_results.csv
